# Лагерные хроники (Stable UI + внешние ассеты)
Оконная версия на tkinter: работает без генерации изображений, с fallback-сценами и поддержкой внешних файлов из папки assets/.

In [ ]:
import os
import tkinter as tk


class Character:
    def __init__(self, name, role):
        self.name = name
        self.role = role


class PlayerState:
    def __init__(self, name):
        self.name = name
        self.energy = 3
        self.reputation = 0
        self.courage = 0
        self.inventory = []
        self.clues = set()
        self.route = []
        self.history = []
        self.ending = ''


class SceneManager:
    def __init__(self, folder='assets'):
        self.folder = folder
        self.scene_info = {
            'camp_gate': {'title': 'Ворота лагеря', 'color': '#79a6d2', 'label': 'Ворота лагеря'},
            'square': {'title': 'Площадь и дневной обход', 'color': '#90b77d', 'label': 'Центральная площадь'},
            'radio': {'title': 'Радиорубка', 'color': '#777799', 'label': 'Радиорубка'},
            'night': {'title': 'Ночной склад', 'color': '#2e3d66', 'label': 'Ночной эпизод'},
            'final': {'title': 'Финальный сбор', 'color': '#b79062', 'label': 'Финальный сбор'}
        }
        self.file_map = {
            'camp_gate': 'camp_gate.png',
            'square': 'square.png',
            'radio': 'radio.png',
            'night': 'night.png',
            'final': 'final.png'
        }

    def image_path(self, key):
        if key not in self.file_map:
            return None
        path = os.path.join(self.folder, self.file_map[key])
        if os.path.exists(path):
            return path
        return None


class GameEngine:
    def __init__(self, player_name):
        self.player = PlayerState(player_name)
        self.characters = [
            Character('Алиса', 'лидер отряда'),
            Character('Электроник', 'техник'),
            Character('Ольга Дмитриевна', 'вожатая')
        ]
        self.locations = {
            'площадь': 'У памятника свежие следы на влажной земле.',
            'сцена': 'За кулисами лежит список выступлений с пометками.',
            'столовая': 'Дежурные шепчутся о ночном силуэте у склада.',
            'лодочная': 'Под досками тайник с ржавым замком.',
            'радиорубка': 'В журнале дежурств не хватает страницы.'
        }
        self.suspicions = {'Алиса': 1, 'Электроник': 2, 'Ольга Дмитриевна': 0}
        self.inspect_order = list(self.locations.keys())
        self.inspect_index = 0
        self.checked_places = 0

    def _remember(self, text):
        self.player.history.append(text)
        self.player.route.append(text)

    def begin(self):
        self._remember('Прибытие в лагерь')
        return 'Лагерь "Солнечная Заря". Ночью пропал архивный ящик с документами. Нужно провести расследование.'

    def apply_day_one(self, choice):
        if choice == 'help':
            self.player.reputation += 2
            self.player.inventory.append('блокнот вожатой')
            self._remember('Помощь вожатой')
            return 'Ты помог(ла) вожатой собрать показания. Репутация выросла.'
        self.player.courage += 2
        self.player.energy -= 1
        self.player.inventory.append('старый компас')
        self._remember('Одиночный поиск')
        return 'Ты пошел(ла) по следу один(одна). Смелость выросла, но сил стало меньше.'

    def has_next_location(self):
        return self.inspect_index < len(self.inspect_order) and self.checked_places < 3

    def next_location(self):
        if not self.has_next_location():
            return None
        return self.inspect_order[self.inspect_index]

    def inspect_location(self, inspect_yes):
        place = self.inspect_order[self.inspect_index]
        self.inspect_index += 1

        if not inspect_yes:
            self._remember('Пропуск локации: ' + place)
            return place, 'Ты пропустил(а) локацию: ' + place

        self.checked_places += 1
        self._remember('Осмотр: ' + place)
        self.player.clues.add(place)
        self.player.inventory.append('улика: ' + place)

        if place == 'лодочная':
            self.player.courage += 1
        if place == 'столовая':
            self.player.reputation += 1

        return place, self.locations[place]

    def radio_attempt(self, code):
        if code == '1989':
            self.player.clues.add('код-1989')
            self.player.reputation += 1
            self._remember('Радиорубка взломана')
            return True, 'Код верный. Найдено имя дежурного из журнала.'
        return False, 'Код неверный.'

    def radio_failed(self):
        self._remember('Радиорубка не взломана')

    def night_action(self, action):
        if action == 'chase':
            self.player.courage += 1
            self.player.energy -= 1
            self.player.clues.add('следы у склада')
            self._remember('Ночное преследование')
            return 'Ты преследовал(а) силуэт и нашел(ла) новые следы.'

        self.player.reputation += 1
        self._remember('Позвал помощь')
        return 'Ты позвал(а) помощь и усилил(а) доверие отряда.'

    def optimize_inventory(self):
        if len(self.player.inventory) > 6:
            self.player.inventory.pop(0)
        if 'старый компас' in self.player.inventory and 'блокнот вожатой' in self.player.inventory:
            self.player.inventory.remove('старый компас')
            self.player.inventory.append('карта маршрутов')
        if 'улика: сцена' in self.player.inventory:
            self.player.clues.add('переписанный сценарий')

    def final_decision(self, accuse):
        if accuse == 'Алиса':
            self.suspicions['Алиса'] += 2
            self._remember('Обвинение Алисы')
        elif accuse == 'Электроник':
            self.suspicions['Электроник'] += 2
            self._remember('Обвинение Электроника')
        else:
            self.player.reputation += 1
            self._remember('Никого не обвинял, анализировал факты')

        many_clues = len(self.player.clues) >= 3
        trusted = self.player.reputation >= 3
        brave = self.player.courage >= 2

        if many_clues and trusted and brave and accuse == 'Никого':
            self.player.ending = 'Истинная концовка: ты раскрываешь подмену архивов и спасаешь смену.'
        elif many_clues and accuse in ('Алиса', 'Электроник'):
            self.player.ending = 'Драматичная концовка: виновный назван, но лагерь расколот.'
        elif trusted or many_clues:
            self.player.ending = 'Нейтральная концовка: часть правды открыта, но не вся.'
        else:
            self.player.ending = 'Плохая концовка: дело закрыли без убедительных доказательств.'

        return self.player.ending

    def build_summary(self):
        summary = {}
        summary['герой'] = self.player.name
        summary['энергия'] = self.player.energy
        summary['репутация'] = self.player.reputation
        summary['смелость'] = self.player.courage
        summary['маршрут'] = self.player.route
        summary['история'] = self.player.history
        summary['улики'] = list(self.player.clues)
        summary['инвентарь'] = self.player.inventory
        summary['подозрения'] = self.suspicions
        summary['финал'] = self.player.ending
        return summary

    def save_summary(self, path='novel_result.txt'):
        s = self.build_summary()
        lines = [
            'Герой: ' + s['герой'],
            'Энергия: ' + str(s['энергия']),
            'Репутация: ' + str(s['репутация']),
            'Смелость: ' + str(s['смелость']),
            'Маршрут:'
        ]
        for step in s['маршрут']:
            lines.append('- ' + step)
        lines.append('History log:')
        for item in s['история']:
            lines.append('* ' + item)
        lines.append('Улики: ' + ', '.join(s['улики']))
        lines.append('Инвентарь: ' + ', '.join(s['инвентарь']))
        lines.append('Подозрения: ' + str(s['подозрения']))
        lines.append('Концовка: ' + s['финал'])
        with open(path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))


class NovelWindow:
    def __init__(self, root):
        self.root = root
        self.root.title('Лагерные хроники — Stable UI')
        self.root.geometry('1100x760')

        self.engine = None
        self.radio_attempts = 3

        self.scenes = SceneManager('assets')
        self.images = {}

        self.header = tk.Label(root, text='Лагерные хроники', font=('Arial', 19, 'bold'))
        self.header.pack(pady=6)

        self.scene_name = tk.Label(root, text='Сцена: старт', font=('Arial', 12, 'bold'))
        self.scene_name.pack()

        self.scene_frame = tk.Frame(root)
        self.scene_frame.pack(pady=6)

        self.scene_canvas = tk.Canvas(self.scene_frame, width=820, height=220, bg='#cfd6df', highlightthickness=1, highlightbackground='#9da6b2')
        self.scene_canvas.pack()

        self.stats_label = tk.Label(root, text='', font=('Arial', 11))
        self.stats_label.pack(pady=3)

        content = tk.Frame(root)
        content.pack(fill='both', expand=True, padx=10, pady=8)

        left = tk.Frame(content)
        left.pack(side='left', fill='both', expand=True)

        right = tk.Frame(content)
        right.pack(side='right', fill='y', padx=(8, 0))

        self.story = tk.Text(left, wrap='word', height=18, width=75, state='disabled')
        self.story.pack(fill='both', expand=True)

        self.controls = tk.Frame(left)
        self.controls.pack(fill='x', pady=8)

        self.name_entry = tk.Entry(self.controls, width=28)
        self.name_entry.grid(row=0, column=0, padx=4)
        self.name_entry.insert(0, 'Семён')

        self.start_button = tk.Button(self.controls, text='Начать игру', command=self.start_game)
        self.start_button.grid(row=0, column=1, padx=4)

        self.action_frame = tk.Frame(left)
        self.action_frame.pack(fill='x', pady=6)

        tk.Label(right, text='History log', font=('Arial', 12, 'bold')).pack(anchor='w')
        self.history_box = tk.Listbox(right, width=42, height=24)
        self.history_box.pack(fill='y')

        self._set_scene('camp_gate')

    def _set_scene(self, key):
        info = self.scenes.scene_info.get(key, {'title': 'Сцена', 'color': '#bbbbbb', 'label': 'Scene'})
        self.scene_name.configure(text='Сцена: ' + info['title'])

        self.scene_canvas.delete('all')
        self.scene_canvas.configure(bg=info['color'])

        image_path = self.scenes.image_path(key)
        if image_path is not None:
            try:
                if key not in self.images:
                    self.images[key] = tk.PhotoImage(file=image_path)
                self.scene_canvas.create_image(410, 110, image=self.images[key])
                return
            except Exception:
                pass

        self.scene_canvas.create_rectangle(30, 25, 790, 195, outline='#ffffff')
        self.scene_canvas.create_text(410, 110, text=info['label'], fill='white', font=('Arial', 18, 'bold'))
        self.scene_canvas.create_text(
            410,
            190,
            text='(положи свое изображение в assets/' + self.scenes.file_map.get(key, '') + ')',
            fill='white',
            font=('Arial', 10)
        )

    def clear_actions(self):
        for w in self.action_frame.winfo_children():
            w.destroy()

    def log(self, text):
        self.story.configure(state='normal')
        self.story.insert('end', text + '\n\n')
        self.story.see('end')
        self.story.configure(state='disabled')

    def add_history(self, text):
        self.history_box.insert('end', text)
        self.history_box.see('end')

    def update_stats(self):
        p = self.engine.player
        self.stats_label.config(
            text='Герой: {0} | Энергия: {1} | Репутация: {2} | Смелость: {3} | Улик: {4}'.format(
                p.name, p.energy, p.reputation, p.courage, len(p.clues)
            )
        )

    def start_game(self):
        name = self.name_entry.get().strip() or 'Семён'
        self.engine = GameEngine(name)
        self.radio_attempts = 3

        self.story.configure(state='normal')
        self.story.delete('1.0', 'end')
        self.story.configure(state='disabled')
        self.history_box.delete(0, 'end')

        self._set_scene('camp_gate')
        intro = self.engine.begin()
        self.log(intro)
        self.add_history('Старт игры: ' + name)
        self.add_history('Событие: Прибытие в лагерь')
        self.update_stats()
        self.show_day_one()

    def show_day_one(self):
        self.clear_actions()
        tk.Button(self.action_frame, text='Помочь вожатой собрать показания', command=lambda: self.day_one_choice('help')).pack(fill='x', pady=3)
        tk.Button(self.action_frame, text='Пойти по следу в одиночку', command=lambda: self.day_one_choice('solo')).pack(fill='x', pady=3)

    def day_one_choice(self, choice):
        msg = self.engine.apply_day_one(choice)
        self.log(msg)
        self.add_history('Решение 1: ' + ('помощь вожатой' if choice == 'help' else 'одиночный поиск'))
        self.update_stats()
        self._set_scene('square')
        self.show_inspection()

    def show_inspection(self):
        self.clear_actions()
        if not self.engine.has_next_location():
            self.log('Осмотр локаций завершен.')
            self.add_history('Этап: осмотр завершен')
            self._set_scene('radio')
            self.show_radio_stage()
            return

        place = self.engine.next_location()
        tk.Label(self.action_frame, text='Осмотреть локацию: ' + place + '?').pack(pady=4)
        tk.Button(self.action_frame, text='Осмотреть', command=lambda: self.inspect_choice(True)).pack(fill='x', pady=3)
        tk.Button(self.action_frame, text='Пропустить', command=lambda: self.inspect_choice(False)).pack(fill='x', pady=3)

    def inspect_choice(self, inspect_yes):
        place, msg = self.engine.inspect_location(inspect_yes)
        self.log(msg)
        self.add_history('Локация ' + place + ': ' + ('осмотрена' if inspect_yes else 'пропущена'))
        self.update_stats()
        self.show_inspection()

    def show_radio_stage(self):
        self.clear_actions()
        tk.Label(self.action_frame, text='Радиорубка: введи 4-значный код (3 попытки)').pack(pady=4)
        self.code_entry = tk.Entry(self.action_frame, width=14)
        self.code_entry.pack(pady=3)
        tk.Button(self.action_frame, text='Проверить код', command=self.submit_code).pack(fill='x', pady=3)

    def submit_code(self):
        code = self.code_entry.get().strip()
        ok, msg = self.engine.radio_attempt(code)
        self.log(msg)

        if ok:
            self.add_history('Радиокод введен верно')
            self.update_stats()
            self._set_scene('night')
            self.show_night_stage()
            return

        self.radio_attempts -= 1
        self.add_history('Радиокод неверный, осталось: ' + str(self.radio_attempts))
        if self.radio_attempts <= 0:
            self.log('Попытки закончились. Радиорубку открыть не удалось.')
            self.engine.radio_failed()
            self._set_scene('night')
            self.show_night_stage()
        else:
            self.log('Осталось попыток: ' + str(self.radio_attempts))

    def show_night_stage(self):
        self.clear_actions()
        tk.Label(self.action_frame, text='Ночной эпизод у склада').pack(pady=4)
        tk.Button(self.action_frame, text='Преследовать силуэт', command=lambda: self.night_choice('chase')).pack(fill='x', pady=3)
        tk.Button(self.action_frame, text='Позвать помощь', command=lambda: self.night_choice('help')).pack(fill='x', pady=3)

    def night_choice(self, action):
        self.log(self.engine.night_action(action))
        self.add_history('Ночной выбор: ' + ('преследование' if action == 'chase' else 'позвать помощь'))
        self.engine.optimize_inventory()
        self.update_stats()
        self._set_scene('final')
        self.show_final_stage()

    def show_final_stage(self):
        self.clear_actions()
        tk.Label(self.action_frame, text='Кого обвинить на общем сборе?').pack(pady=4)
        tk.Button(self.action_frame, text='Обвинить Алису', command=lambda: self.finish_game('Алиса')).pack(fill='x', pady=3)
        tk.Button(self.action_frame, text='Обвинить Электроника', command=lambda: self.finish_game('Электроник')).pack(fill='x', pady=3)
        tk.Button(self.action_frame, text='Никого не обвинять, опереться на факты', command=lambda: self.finish_game('Никого')).pack(fill='x', pady=3)

    def finish_game(self, accuse):
        ending = self.engine.final_decision(accuse)
        self.engine.save_summary('novel_result.txt')
        self.update_stats()
        self.clear_actions()
        self.log('Финал: ' + ending)
        self.log('Итог сохранен в novel_result.txt')
        self.add_history('Финальное решение: ' + accuse)
        self.add_history('Концовка: ' + ending)
        tk.Button(self.action_frame, text='Начать заново', command=self.start_game).pack(fill='x', pady=3)


root = tk.Tk()
app = NovelWindow(root)
root.mainloop()
